<a href="https://colab.research.google.com/github/Riley-McDonald/Airline-Case-Study-Problems/blob/main/FlightCase2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

It's Monday morning. The Director of System Operations walks into the SOC Weekly Ops Review and asks: "Which routes are consistently killing our on-time performance, and does time of day matter?"

In [18]:
#import pandas and numpy, csv file

import pandas as pd
import numpy as np

df = pd.read_csv("case2_flight_ops.csv", parse_dates=['flight_date'])
df.head()

,flight_date,flight_num,origin,dest,dep_hour,arr_delay_min
0,2024-01-08,B6 415,JFK,LAX,6,42
1,2024-01-08,B6 416,JFK,LAX,6,38
2,2024-01-08,B6 101,JFK,BOS,7,5
3,2024-01-08,B6 102,JFK,BOS,7,8
4,2024-01-08,B6 200,JFK,FLL,8,12


In [19]:
#check for nulls

df.isna().sum()

,0
flight_date,0
flight_num,0
origin,0
dest,0
dep_hour,0
arr_delay_min,0


In [20]:
#check for duplicate data

duplicate_count = df.duplicated().sum()
print(duplicate_count)

0


In [21]:
#read data info

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   flight_date    50 non-null     datetime64[ns]
 1   flight_num     50 non-null     object        
 2   origin         50 non-null     object        
 3   dest           50 non-null     object        
 4   dep_hour       50 non-null     int64         
 5   arr_delay_min  50 non-null     int64         
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 2.5+ KB


In [22]:
df.head()

,flight_date,flight_num,origin,dest,dep_hour,arr_delay_min
0,2024-01-08,B6 415,JFK,LAX,6,42
1,2024-01-08,B6 416,JFK,LAX,6,38
2,2024-01-08,B6 101,JFK,BOS,7,5
3,2024-01-08,B6 102,JFK,BOS,7,8
4,2024-01-08,B6 200,JFK,FLL,8,12


In [23]:
#make a delayed column for true false

df['delayed'] = df['arr_delay_min'] > 15
df.head()

,flight_date,flight_num,origin,dest,dep_hour,arr_delay_min,delayed
0,2024-01-08,B6 415,JFK,LAX,6,42,True
1,2024-01-08,B6 416,JFK,LAX,6,38,True
2,2024-01-08,B6 101,JFK,BOS,7,5,False
3,2024-01-08,B6 102,JFK,BOS,7,8,False
4,2024-01-08,B6 200,JFK,FLL,8,12,False


In [24]:
#findings max and min numbers for departure hour

print(df['dep_hour'].min())
print(df['dep_hour'].max())

6
23


In [25]:
# make column for time of day a flight is leaving

conditions = [
    df['dep_hour'] <= 12,
    df['dep_hour'] <=17,
    df['dep_hour'] <= 21,
]

choices = ['morning','afternoon','evening']

df['time_of_day'] = np.select(conditions, choices, default='red_eye')
df.head()

,flight_date,flight_num,origin,dest,dep_hour,arr_delay_min,delayed,time_of_day
0,2024-01-08,B6 415,JFK,LAX,6,42,True,morning
1,2024-01-08,B6 416,JFK,LAX,6,38,True,morning
2,2024-01-08,B6 101,JFK,BOS,7,5,False,morning
3,2024-01-08,B6 102,JFK,BOS,7,8,False,morning
4,2024-01-08,B6 200,JFK,FLL,8,12,False,morning


In [26]:
# make a dataframe consisting of amount of flights delayed and
# amount of flights flown per route per time day

grouped = df.groupby(['origin','dest','time_of_day']).agg(
    amount_delayed = ('delayed','sum'),
    total_flights = ('delayed','count')
).reset_index()

print(grouped)

   origin dest time_of_day  amount_delayed  total_flights
0     BOS  JFK     red_eye               3              3
1     JFK  BOS   afternoon               0              4
2     JFK  BOS     morning               0              5
3     JFK  FLL   afternoon               5              5
4     JFK  FLL     morning               0              4
5     JFK  LAX   afternoon               8              8
6     JFK  LAX     evening               0              3
7     JFK  LAX     morning               4              5
8     JFK  MCO     evening               4              4
9     JFK  MCO     morning               4              5
10    JFK  SFO     morning               0              4


In [27]:
#calculate otp percentage columns

grouped['otp_pct']= ((grouped['total_flights']-grouped['amount_delayed'])/grouped['total_flights']*100).round(1)
grouped.head(20)

,origin,dest,time_of_day,amount_delayed,total_flights,otp_pct
0,BOS,JFK,red_eye,3,3,0.0
1,JFK,BOS,afternoon,0,4,100.0
2,JFK,BOS,morning,0,5,100.0
3,JFK,FLL,afternoon,5,5,0.0
4,JFK,FLL,morning,0,4,100.0
5,JFK,LAX,afternoon,8,8,0.0
6,JFK,LAX,evening,0,3,100.0
7,JFK,LAX,morning,4,5,20.0
8,JFK,MCO,evening,4,4,0.0
9,JFK,MCO,morning,4,5,20.0


In [28]:
# make df to group routes and list their mean otp_pct
# add rank to rank the otp by route

route_summary = grouped.groupby(['origin','dest'])['otp_pct'].mean().reset_index()
route_summary['rank'] = route_summary['otp_pct'].rank(ascending=True)
print(route_summary.sort_values('rank'))

  origin dest  otp_pct  rank
0    BOS  JFK      0.0   1.0
4    JFK  MCO     10.0   2.0
3    JFK  LAX     40.0   3.0
2    JFK  FLL     50.0   4.0
1    JFK  BOS    100.0   5.5
5    JFK  SFO    100.0   5.5


In [29]:
# look at how otp shifts by time of day for each route

time_pivot = grouped.pivot_table(index=['origin','dest'], columns='time_of_day', values ='otp_pct')
print(time_pivot)

time_of_day  afternoon  evening  morning  red_eye
origin dest                                      
BOS    JFK         NaN      NaN      NaN      0.0
JFK    BOS       100.0      NaN    100.0      NaN
       FLL         0.0      NaN    100.0      NaN
       LAX         0.0    100.0     20.0      NaN
       MCO         NaN      0.0     20.0      NaN
       SFO         NaN      NaN    100.0      NaN
